# Automated Babesia Infection Detection from Blood Sample Images
### CNN + SoftMax Classifier — End-to-End Colab Notebook

**Project:** Automated Babesia Infection Detection from Blood Sample Images Using CNN and SoftMax Classification
**Team:** Jai Ganesh, Gunalan V, Deepak R, Naveen Kumar A — SIMATS Engineering (MLA0409, Deep Learning for Applications)

---

### ⚠️ Important note about the dataset — please read before you run this

There is **no public, labelled image dataset of human Babesia-infected blood smears** large enough to train a CNN (Babesia is rare in humans and almost all public image datasets are for veterinary/canine Babesia, which are not directly usable here). Your own literature review (Slide 5 / Section 2.2 of your report) makes exactly this point: **Babesia parasites look almost identical to Malaria parasites under the microscope** — both are intra‑erythrocytic protozoan parasites that produce the same kind of ring‑shaped inclusions inside red blood cells.

Because of this, the standard (and academically accepted) practice for student/capstone projects on Babesia — when no Babesia image bank exists — is to **train and validate the CNN + SoftMax pipeline on the public NIH/Kaggle Malaria Blood Smear dataset** (`iarunava/cell-images-for-detecting-malaria`, 27,558 thin blood‑smear cell images, *Parasitized* vs *Uninfected*), and present it as a **proof-of-concept pipeline for "intra-erythrocytic parasite detection"** that is directly transferable to Babesia once a labelled Babesia image set becomes available (this is literally listed as a "Possible Improvement" in your report — *Section 4.3, Larger Datasets*).

This notebook:
- Downloads that dataset **automatically** — no manual upload, no Kaggle account, no API key. It uses `tensorflow_datasets`, which mirrors the exact same NIH dataset that is hosted on Kaggle, directly from Google's public storage.
- Renames the two classes to **`Infected`** and **`Normal`** in all outputs/plots, so the notebook reads exactly like your report and slides.
- Is written so that the **only thing you change later is the data source** — if your guide/college gives you an actual Babesia smear dataset, drop it into the same folder structure (`Infected/`, `Normal/`) and everything downstream (model, training, evaluation, deployment) works unchanged. The code cell in **Section 1B** shows exactly how to do that swap.

Be upfront about this with your guide/panel — say you used the malaria dataset as a stand‑in proxy for Babesia because of the documented visual similarity and lack of public Babesia data. This is a legitimate, common approach and reviewers are very used to seeing it.

---


## 0. Setup — Runtime, GPU, and Libraries
Run this first. In Colab: **Runtime → Change runtime type → T4 GPU** before running, for fast training.

In [ ]:
# Check GPU
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPU available:", len(gpus) > 0, "-", gpus)


In [ ]:
# Install/upgrade the few packages Colab doesn't already have at the right version
!pip install -q tensorflow-datasets scikit-learn seaborn opencv-python-headless --upgrade


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers, models, callbacks, optimizers
from sklearn.metrics import (confusion_matrix, classification_report,
                              roc_curve, auc, ConfusionMatrixDisplay)

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

CLASS_NAMES = ["Normal", "Infected"]   # 0 = Normal (Uninfected), 1 = Infected (Parasitized)
IMG_SIZE = 128                          # resize every image to 128x128
BATCH_SIZE = 32
EPOCHS = 20                             # raise to 30-40 once you confirm the pipeline works end-to-end


## 1A. Load the Dataset (automatic download — no Kaggle key, no manual upload)

`tensorflow_datasets` downloads the official NIH malaria blood-smear dataset directly (the same dataset published on Kaggle as `iarunava/cell-images-for-detecting-malaria`) and caches it inside the Colab VM. This step only needs to run once per session — it takes a few minutes the first time (~330 MB).

In [ ]:
# Load with an explicit train/val/test split (the raw dataset only ships a single "train" split)
(ds_train, ds_val, ds_test), ds_info = tfds.load(
    "malaria",
    split=["train[:80%]", "train[80%:90%]", "train[90%:]"],
    as_supervised=True,     # returns (image, label) tuples
    with_info=True,
    shuffle_files=True,
)

print(ds_info.description)
print("\nLabel names in source dataset:", ds_info.features["label"].names)
# Source labels: 0 = parasitized (Infected), 1 = uninfected (Normal) -> we will remap below
print("Total images:", ds_info.splits["train"].num_examples)


In [ ]:
# IMPORTANT: tfds 'malaria' encodes label 0 = parasitized, 1 = uninfected.
# We remap so that 1 = Infected, 0 = Normal everywhere in THIS notebook,
# matching the class order used in the evaluation section and your report.
def remap_label(image, label):
    new_label = 1 - label   # flip: source 0(parasitized)->1(Infected), source 1(uninfected)->0(Normal)
    return image, new_label

ds_train = ds_train.map(remap_label)
ds_val   = ds_val.map(remap_label)
ds_test  = ds_test.map(remap_label)


### 1B. (Optional) Swap in a real Babesia dataset later

If you later get an actual labelled Babesia blood-smear dataset, organise it as:
```
babesia_dataset/
    Infected/   *.png / *.jpg
    Normal/     *.png / *.jpg
```
and replace the loading cell above with:
```python
data_dir = "/content/babesia_dataset"
full_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, labels="inferred", label_mode="int",
    class_names=["Normal", "Infected"], image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=None, shuffle=True, seed=SEED)
n = full_ds.cardinality().numpy()
ds_train = full_ds.take(int(0.8*n))
ds_val   = full_ds.skip(int(0.8*n)).take(int(0.1*n))
ds_test  = full_ds.skip(int(0.9*n))
```
Everything from Section 2 onward (preprocessing, model, training, evaluation, deployment) needs **no changes** — it works on any dataset that yields `(image, label)` pairs with labels `0=Normal, 1=Infected`.

## 2. Preprocessing & Data Augmentation
Matches Module 1 of your project (resizing, normalisation, noise handling, augmentation, train/val/test split).

In [ ]:
def preprocess(image, label, augment=False):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32) / 255.0          # normalisation
    if augment:
        image = tf.image.random_flip_left_right(image)
        image = tf.image.random_flip_up_down(image)
        image = tf.image.random_brightness(image, max_delta=0.1)
        image = tf.image.random_contrast(image, lower=0.9, upper=1.1)
        image = tf.image.rot90(image, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
    return image, label

def make_pipeline(ds, augment=False, shuffle=False):
    if shuffle:
        ds = ds.shuffle(2000, seed=SEED)
    ds = ds.map(lambda x, y: preprocess(x, y, augment), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_pipe = make_pipeline(ds_train, augment=True, shuffle=True)
val_pipe   = make_pipeline(ds_val,   augment=False)
test_pipe  = make_pipeline(ds_test,  augment=False)


In [ ]:
# Visual sanity check — a few samples after preprocessing
plt.figure(figsize=(10, 6))
for images, labels in train_pipe.take(1):
    for i in range(8):
        plt.subplot(2, 4, i + 1)
        plt.imshow(images[i])
        plt.title(CLASS_NAMES[int(labels[i])])
        plt.axis("off")
plt.suptitle("Sample preprocessed blood-smear cell images")
plt.tight_layout()
plt.show()


## 3. CNN + SoftMax Model (Module 2/3/4 of your project)
A custom CNN feature extractor followed by a SoftMax classification head, exactly as described in your architecture diagram.

In [ ]:
USE_TRANSFER_LEARNING = False   # set True to use a MobileNetV2 backbone instead (usually higher accuracy, similar speed on GPU)

def build_custom_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=2):
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="softmax_output")(x)

    return models.Model(inputs, outputs, name="Babesia_CNN_SoftMax")


def build_transfer_model(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=2):
    base = tf.keras.applications.MobileNetV2(input_shape=input_shape, include_top=False, weights="imagenet")
    base.trainable = False
    inputs = layers.Input(shape=input_shape)
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="softmax_output")(x)
    return models.Model(inputs, outputs, name="Babesia_MobileNetV2_SoftMax")


model = build_transfer_model() if USE_TRANSFER_LEARNING else build_custom_cnn()
model.summary()


In [ ]:
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",   # labels are plain ints (0/1), output is 2-way softmax
    metrics=["accuracy"],
)


## 4. Training (Module 5: Integrated Model Training & Validation)

In [ ]:
checkpoint_path = "best_babesia_model.keras"

cb = [
    callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
    callbacks.ModelCheckpoint(checkpoint_path, monitor="val_accuracy", save_best_only=True),
]

history = model.fit(
    train_pipe,
    validation_data=val_pipe,
    epochs=EPOCHS,
    callbacks=cb,
)


In [ ]:
# Training curves
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history["accuracy"], label="Train")
ax[0].plot(history.history["val_accuracy"], label="Validation")
ax[0].set_title("Accuracy"); ax[0].set_xlabel("Epoch"); ax[0].legend()

ax[1].plot(history.history["loss"], label="Train")
ax[1].plot(history.history["val_loss"], label="Validation")
ax[1].set_title("Loss"); ax[1].set_xlabel("Epoch"); ax[1].legend()
plt.tight_layout()
plt.show()


## 5. Evaluation — Accuracy, Sensitivity, Specificity, ROC-AUC, Confusion Matrix
These are exactly the metrics listed in Chapter 4 of your report.

In [ ]:
# Load the best checkpoint (in case training continued past the best epoch)
model = tf.keras.models.load_model(checkpoint_path)

test_loss, test_acc = model.evaluate(test_pipe)
print(f"Test Accuracy: {test_acc*100:.2f}%")


In [ ]:
# Collect predictions over the full test set
y_true, y_prob = [], []
for images, labels in test_pipe:
    probs = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_prob.extend(probs[:, 1])   # probability of class "Infected"

y_true = np.array(y_true)
y_prob = np.array(y_prob)
y_pred = (y_prob >= 0.5).astype(int)

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix — Babesia (proxy) Detection")
plt.show()

tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)   # = Recall for "Infected"
specificity = tn / (tn + fp)
print(f"Sensitivity (Recall, Infected): {sensitivity:.4f}")
print(f"Specificity (Recall, Normal):  {specificity:.4f}")


In [ ]:
# ROC curve + AUC
fpr, tpr, _ = roc_curve(y_true, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], "k--", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.show()


In [ ]:
# Visual check: a grid of test images with predicted vs true label
plt.figure(figsize=(12, 8))
for images, labels in test_pipe.take(1):
    probs = model.predict(images, verbose=0)
    preds = np.argmax(probs, axis=1)
    for i in range(12):
        plt.subplot(3, 4, i + 1)
        plt.imshow(images[i])
        true_l = CLASS_NAMES[int(labels[i])]
        pred_l = CLASS_NAMES[preds[i]]
        conf = probs[i][preds[i]] * 100
        color = "green" if true_l == pred_l else "red"
        plt.title(f"True: {true_l}\nPred: {pred_l} ({conf:.1f}%)", color=color, fontsize=9)
        plt.axis("off")
plt.tight_layout()
plt.show()


## 6. Explainability — Grad-CAM (matches "Possible Improvements", Section 4.3 of your report)
Shows which regions of the cell image the CNN focused on, useful for the viva / demo.

In [ ]:
def get_last_conv_layer(model):
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            return layer.name
    raise ValueError("No Conv2D layer found")

def grad_cam(model, image, class_index, last_conv_layer_name):
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(image[np.newaxis, ...])
        loss = predictions[:, class_index]
    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

if not USE_TRANSFER_LEARNING:
    last_conv = get_last_conv_layer(model)
    for images, labels in test_pipe.take(1):
        img = images[0].numpy()
        true_label = int(labels[0])
        pred_class = int(np.argmax(model.predict(img[np.newaxis, ...], verbose=0)))
        heatmap = grad_cam(model, img, pred_class, last_conv)
        heatmap_resized = tf.image.resize(heatmap[..., np.newaxis], (IMG_SIZE, IMG_SIZE)).numpy().squeeze()

        fig, ax = plt.subplots(1, 2, figsize=(8, 4))
        ax[0].imshow(img); ax[0].set_title(f"Original ({CLASS_NAMES[true_label]})"); ax[0].axis("off")
        ax[1].imshow(img); ax[1].imshow(heatmap_resized, cmap="jet", alpha=0.5)
        ax[1].set_title(f"Grad-CAM (Pred: {CLASS_NAMES[pred_class]})"); ax[1].axis("off")
        plt.tight_layout(); plt.show()
        break
else:
    print("Grad-CAM cell above targets the custom CNN's Conv2D layers directly. "
          "Set USE_TRANSFER_LEARNING=False and re-train to view Grad-CAM, or adapt the "
          "layer name to a MobileNetV2 block if you want this for the transfer-learning model.")


## 7. Save the Trained Model
Saves in the modern Keras format. Download it — you'll need this exact file for the deployment app in Section 8.

In [ ]:
FINAL_MODEL_PATH = "babesia_cnn_softmax_model.keras"
model.save(FINAL_MODEL_PATH)
print("Saved:", FINAL_MODEL_PATH)

from google.colab import files
files.download(FINAL_MODEL_PATH)   # triggers a browser download


In [ ]:
# (Optional) Single-image inference helper — useful for quickly testing one image
# before wiring up the deployment app below.
def predict_single_image(model, image_path, img_size=IMG_SIZE):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_image(img, channels=3)
    img = tf.image.resize(img, (img_size, img_size))
    img = tf.cast(img, tf.float32) / 255.0
    img = tf.expand_dims(img, 0)
    probs = model.predict(img, verbose=0)[0]
    pred_idx = int(np.argmax(probs))
    return CLASS_NAMES[pred_idx], float(probs[pred_idx]), probs

# Example (uncomment and point to a real file once you've uploaded one to the Colab session):
# label, confidence, probs = predict_single_image(model, "/content/sample_cell.png")
# print(f"Prediction: {label}  |  Confidence: {confidence*100:.2f}%  |  Raw probs: {probs}")


## 8. Deployment — Where and How

A trained `.keras` model file by itself isn't something a doctor, your panel, or a classmate can "use" — it needs a small app wrapped around it. For a capstone/academic project, the simplest, **completely free**, and most demo-friendly route is a **Streamlit web app deployed on Streamlit Community Cloud**. Below is the exact plan.

### Recommended stack
| Layer | Choice | Why |
|---|---|---|
| App framework | **Streamlit** | A working "upload image → get prediction" UI in ~40 lines of Python, no HTML/CSS/JS needed |
| Hosting | **Streamlit Community Cloud** (share.streamlit.io) | Free forever for public apps, deploys straight from a GitHub repo, gives you a public `https://...streamlit.app` link for your viva/demo |
| Code hosting | **GitHub** | Streamlit Cloud deploys directly from a repo |

### Step-by-step
1. **Download** `babesia_cnn_softmax_model.keras` from Section 7 above (and the `app.py` + `requirements.txt` files provided alongside this notebook).
2. Create a new **public GitHub repository**, e.g. `babesia-detector-app`.
3. Push these 3 files to the repo root:
   - `app.py`
   - `requirements.txt`
   - `babesia_cnn_softmax_model.keras`  (if it's over ~95 MB, see the size note below)
4. Go to **share.streamlit.io** → sign in with GitHub → **"New app"** → pick your repo, branch `main`, file `app.py` → **Deploy**.
5. Wait 2-3 minutes for the build. You'll get a public URL — open it, upload a blood-smear cell image, and it will show **Infected / Normal** with a confidence score. Use this live link in your final demo/viva.

### If the model file is too large for GitHub (>100 MB hard limit)
- Use **Git LFS** (`git lfs track "*.keras"`) — works up to ~1-2 GB on free tier, or
- Host the model file on **Google Drive / Hugging Face Hub** and have `app.py` download it once at startup with `gdown` or `huggingface_hub`, instead of committing the binary to git. (The MobileNetV2 transfer-learning model is much smaller than the custom CNN if size becomes a problem — switch `USE_TRANSFER_LEARNING = True` and retrain.)

### Alternative options (if you want something different from Streamlit)
- **Gradio + Hugging Face Spaces** — equally free, equally simple, slightly more popular specifically for ML model demos; same overall idea (`app.py` using `gr.Interface`, pushed to a HF Space instead of GitHub).
- **Flask/FastAPI + Render or Railway free tier** — gives you a real REST API endpoint (`POST /predict`) instead of just a UI; more "production-engineering" looking for a report, but more setup work and the free tiers sleep after inactivity.
- For a **college LAN/lab demo only** (no public link needed): just run `streamlit run app.py` locally on a laptop with the model file alongside it — zero deployment needed.

For a B.Tech capstone viva, **Streamlit Community Cloud is the right balance of "actually deployed," free, and quick to set up** — go with that unless your guide specifically asks for a REST API.
